# Auditoria de Execuções — Observabilidade Analítica

Notebook de análise sobre `observability.pipeline_runs`, complementar ao registro bruto de
execução já existente. Enquanto os notebooks do pipeline apenas *gravam* eventos, este
notebook *analisa* o histórico em busca de padrões que merecem explicação — execuções
duplicadas, gaps de execução esperada, e duração fora do padrão.

**Execução automática (Task no Job) para manter o Dashboard sempre atualizado — também pode
ser executado manualmente a qualquer momento** para checagem pontual.

Detecção é automática; **causa raiz continua sendo investigação humana** — as tabelas de saída
têm uma coluna `causa_raiz` em branco, preenchida manualmente após investigação (como fizemos
para os casos de 28/08 e 01/09). Gravação via inserção apenas de registros novos
(`inserir_se_novo`), preservando qualquer anotação já feita em execuções anteriores.

**Entrada:** tabela `poc_b3_modernizacao.observability.pipeline_runs`
**Saída:** tabelas `observability.auditoria_anomalias` e `observability.auditoria_gaps`

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# execucao principal - detecta anomalias, grava (so novos registros), registra observabilidade
try:
    df_runs = spark.table("poc_b3_modernizacao.observability.pipeline_runs")

    # deteccao 1 - execucoes proximas (mesmo notebook, mesmo dia, <30 min de diferenca)
    janela_ordenada = Window.partitionBy("notebook", "data_referencia").orderBy("inicio")
    df_com_intervalo = df_runs.withColumn(
        "inicio_anterior", F.lag("inicio").over(janela_ordenada)
    ).withColumn(
        "minutos_desde_anterior",
        (F.col("inicio").cast("long") - F.col("inicio_anterior").cast("long")) / 60
    )
    df_execucoes_proximas = (df_com_intervalo
        .filter((F.col("minutos_desde_anterior").isNotNull()) & (F.col("minutos_desde_anterior") < 30))
        .select(
            "notebook", "data_referencia", "inicio",
            F.lit("execucao_proxima").alias("tipo_anomalia"),
            F.concat(F.round(F.col("minutos_desde_anterior"), 2), F.lit(" min desde a execucao anterior")).alias("detalhe"),
        )
    )

    # deteccao 2 - gaps de execucao esperada (dias uteis sem registro de ingestao)
    datas_com_execucao = (df_runs
        .filter(F.col("notebook") == "01_ingestao_landing")
        .select(F.col("data_referencia").cast("date").alias("data"))
        .distinct()
    )
    data_minima = datas_com_execucao.agg(F.min("data")).collect()[0][0]
    data_maxima = datas_com_execucao.agg(F.max("data")).collect()[0][0]
    todas_as_datas = spark.sql(f"""
        SELECT explode(sequence(to_date('{data_minima}'), to_date('{data_maxima}'), interval 1 day)) as data
    """).filter(F.dayofweek("data").between(2, 6))
    df_gaps = (todas_as_datas.join(datas_com_execucao, on="data", how="left_anti")
        .withColumn("notebook_esperado", F.lit("01_ingestao_landing"))
    )

    # deteccao 3 - duracao fora do padrao (desvio em relacao a media do proprio notebook)
    janela_notebook = Window.partitionBy("notebook")
    df_com_stats = df_runs.withColumn(
        "duracao_media_notebook", F.avg("duracao_segundos").over(janela_notebook)
    ).withColumn(
        "duracao_desvio_padrao_notebook", F.stddev("duracao_segundos").over(janela_notebook)
    ).withColumn(
        "desvios_da_media",
        F.when(F.col("duracao_desvio_padrao_notebook") > 0,
               (F.col("duracao_segundos") - F.col("duracao_media_notebook")) / F.col("duracao_desvio_padrao_notebook"))
         .otherwise(0)
    )
    df_duracao_anormal = (df_com_stats
        .filter(F.abs(F.col("desvios_da_media")) > 2)
        .select(
            "notebook", "data_referencia", "inicio",
            F.lit("duracao_anormal").alias("tipo_anomalia"),
            F.concat(
                F.round(F.col("duracao_segundos"), 2), F.lit("s vs media "),
                F.round(F.col("duracao_media_notebook"), 2), F.lit("s ("),
                F.round(F.col("desvios_da_media"), 2), F.lit(" desvios-padrao)")
            ).alias("detalhe"),
        )
    )

    # unifica as duas anomalias, adiciona causa_raiz vazia e timestamp de deteccao
    df_anomalias = (df_execucoes_proximas.unionByName(df_duracao_anormal)
        .withColumn("causa_raiz", F.lit(None).cast("string"))
        .withColumn("data_deteccao", F.current_timestamp())
    )
    df_gaps_final = (df_gaps
        .withColumn("causa_raiz", F.lit(None).cast("string"))
        .withColumn("data_deteccao", F.current_timestamp())
    )

    print(f"Anomalias detectadas nesta execucao: {df_anomalias.count()}")
    print(f"Gaps detectados nesta execucao: {df_gaps_final.count()}")

    inserir_se_novo(df_anomalias, "poc_b3_modernizacao.observability.auditoria_anomalias", ["notebook", "inicio", "tipo_anomalia"])
    inserir_se_novo(df_gaps_final, "poc_b3_modernizacao.observability.auditoria_gaps", ["data", "notebook_esperado"])

    registrar_execucao(
        notebook="07_auditoria_execucoes",
        data_referencia=data_maxima.strftime("%Y-%m-%d"),
        modo_execucao="reprocessamento_manual",
        status="sucesso",
        inicio=inicio_execucao,
        fim=datetime.now(),
    )

except Exception as e:
    registrar_execucao(
        notebook="07_auditoria_execucoes",
        data_referencia=None,
        modo_execucao="reprocessamento_manual",
        status="falha",
        inicio=inicio_execucao,
        fim=datetime.now(),
        mensagem_erro=str(e),
    )
    raise

In [0]:
# preenchimento manual da causa raiz - investigacao humana, nao automatizada

# grupo 1: reprocessamentos durante desenvolvimento do indice acumulado (28/08, ~14h27-14h32)
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'Reprocessamentos manuais durante o desenvolvimento do indice acumulado (ADR-09) no mesmo dia - desenvolvimento normal, nao anomalia de producao.'
    WHERE notebook = '04_gold'
      AND inicio >= to_timestamp('2026-08-28T14:00:00') AND inicio <= to_timestamp('2026-08-28T15:00:00')
""")

# grupo 2: reprocessamento completo por sincronizacao tardia do KNIME (28/08, ~20h28-20h32 UTC)
# sem filtro por data_referencia, pois a reconciliacao registra a data que ELA reconciliou (27/08),
# nao o dia em que rodou (28/08) - filtramos so pelo horario real da execucao
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'Reprocessamento manual completo motivado pela sincronizacao tardia do CSV do KNIME nesse dia (conflito de merge no Git, resolvido apos as 17h15 local). O Widget modo_execucao manteve o valor "agendado" por nao ter sido alterado manualmente antes do reprocessamento - limitacao conhecida do campo.'
    WHERE inicio >= to_timestamp('2026-08-28T20:28:00') AND inicio <= to_timestamp('2026-08-28T20:32:00')
""")

# grupo 3: testes controlados do try/except (01/09, 18h18-18h39)
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'Testes controlados durante a implementacao do try/except (ADR-13) - sucesso e falha propositais, revertidos ao final.'
    WHERE inicio >= to_timestamp('2026-09-01T18:18:00') AND inicio <= to_timestamp('2026-09-01T18:40:00')
""")

# grupo 4: duracao anormal sem causa confirmada (01/09, ~19h39)
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'NAO DETERMINADA - confirmado pelo autor como execucao manual perceptivelmente mais lenta que o padrao, sem registro do motivo especifico (possivel latencia de rede pontual da API brapi.dev, nao confirmada). Requer atencao, sem explicacao confirmada.'
    WHERE notebook = '01_ingestao_landing' AND tipo_anomalia = 'duracao_anormal'
      AND inicio >= to_timestamp('2026-09-01T19:00:00') AND inicio <= to_timestamp('2026-09-01T20:00:00')
""")

print("Causas raiz preenchidas.")

In [0]:
display(spark.table("poc_b3_modernizacao.observability.auditoria_anomalias").orderBy("inicio"))

In [0]:
display(spark.table("poc_b3_modernizacao.observability.auditoria_anomalias").orderBy("data_deteccao", "inicio"))